# **Model Experiments**

## **Load Dataset & Dependencies**

In [ ]:
import warnings

import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

from src.config import load_config
from src.train import temporal_split, build_candidates, compare_candidates, tune_winner
from src.economics import select_threshold
from src.evaluate import evaluate_baselines, generate_cv_metrics, evaluate_on_test, evaluate_on_unseen_customers, build_comparison_table

In [2]:
config = load_config()
df = pd.read_parquet(config.data.processed_path)

test_cutoff = config.model.test_cutoff
random_state = config.model.random_state
lookback_days = config.labelling.lookback_days

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17857 entries, 0 to 17856
Data columns (total 26 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   customer_id              17857 non-null  int64         
 1   cutoff                   17857 non-null  datetime64[us]
 2   churn_horizon            17857 non-null  int64         
 3   recency                  17857 non-null  int64         
 4   frequency                17857 non-null  int64         
 5   monetary                 17857 non-null  float64       
 6   mean_order_value         17857 non-null  float64       
 7   tenure                   17857 non-null  int64         
 8   avg_inter_purchase_gap   13775 non-null  float64       
 9   std_inter_purchase_gap   10632 non-null  float64       
 10  recency_ratio            13576 non-null  float64       
 11  spend_30d                17857 non-null  float64       
 12  spend_90d                17857 non-null  fl

## **Dataset Splitting & Cross-Validation Strategy Preparation**

### **```temporal_split()``` Prototype and its Design Implications**

Below is the original implementation of ```temporal_split()```. Originally, the function purged customers that appeared in both the training and testing sets to avoid potential data leakage. 

However, when the function was tested in this notebook, concerning metrics were found in the purged report. Firstly, 1827 customers were purged; this was almost half of the training set's customer count. Secondly, the training set churned rate rose to 80.09%. 

These metrics suggested that customers who appeared in both sets likely made up the majority of customers who did not churn in the training set. Purging the data would ultimately result in the model overpredicting churn, due to the significant churn rate disparity between both datasets.

Given that the customer identifiers ("customer_id" and "cutoff") are dropped when building X_train and X_test, and the different cutoffs would result in ultimately different feature values (despite overlapping activity history), the overlapping customers in the training and test sets will still be treated as different entries by the model while fitting, with no actual data leakage occurring. As such, the purging element of ```temporal_split()``` was removed.

In [4]:
def prototype_temporal_split(
    df_processed: pd.DataFrame, test_cutoff: str
) -> tuple[pd.DataFrame, pd.Series, pd.Series, pd.DataFrame, pd.Series]:
    """
    Splits the processed dataset into X and y train and test sets (with a 'groups' identifier series for the train set) 
    based on the test cutoff date.
    """
    df = df_processed.copy()
    cutoff = pd.Timestamp(test_cutoff)

    # Separates train and test sets by test cutoff date
    df_train = df[df["cutoff"] < cutoff]
    df_test = df[df["cutoff"] == cutoff]

    # Purges customers who appear in both sets from the train set and generates 'purged' report
    overlap = set(df_train["customer_id"]) & set(df_test["customer_id"])
    purged = {
        "df_train_initial_rows": len(df_train),
        "df_train_initial_customers": df_train["customer_id"].nunique(),
        "df_test_rows": len(df_test),
        "df_test_customers": df_test["customer_id"].nunique(),
        "purged": len(overlap),
    }
    df_train = df_train[~df_train["customer_id"].isin(overlap)]
    purged["df_train_final_rows"] = len(df_train)
    purged["df_train_final_customers"] = df_train["customer_id"].nunique()
    purged["train_churn_rate"] = round(float(df_train["churned"].mean()), 4)
    purged["test_churn_rate"] = round(float(df_test["churned"].mean()), 4)

    # Splits train set into X_train, y_train and groups
    X_train = df_train.drop(columns=["customer_id", "cutoff", "churn_horizon", "churned"])
    y_train = df_train["churned"]
    groups = df_train["customer_id"]

    # Splits test set into X_test and y_test
    X_test = df_test.drop(columns=["customer_id", "cutoff", "churn_horizon", "churned"])
    y_test = df_test["churned"]    

    return X_train, y_train, groups, X_test, y_test, purged


In [5]:
X_train, y_train, groups, X_test, y_test, purged = prototype_temporal_split(df, test_cutoff)
purged

{'df_train_initial_rows': 15311,
 'df_train_initial_customers': 3836,
 'df_test_rows': 2546,
 'df_test_customers': 2546,
 'purged': 1827,
 'df_train_final_rows': 7342,
 'df_train_final_customers': 2009,
 'train_churn_rate': 0.8009,
 'test_churn_rate': 0.3044}

### **Version Implemented in ```train.py```**

In [6]:
X_train, y_train, groups, X_test, y_test, split_report = temporal_split(df, test_cutoff)
split_report

{'df_train_rows': 15311,
 'df_train_customers': 3836,
 'df_test_rows': 2546,
 'df_test_customers': 2546,
 'overlapping_customers': 1827}

In [7]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=random_state)

## **Generating Models**

In [8]:
candidates = build_candidates(lookback_days=lookback_days, random_state=random_state)

In [9]:
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
    results = compare_candidates(candidates, X_train, y_train, groups, cv)
results

,roc_auc,roc_auc_std,pr_auc,pr_auc_std,pr_auc_base_rate,train_roc_auc,fit_seconds
model,,,,,,,
logistic,0.765094,0.007435,0.678606,0.013739,0.448566,0.770302,0.039547
random_forest,0.758152,0.010047,0.673083,0.015633,0.448566,0.972167,0.449265
gradient_boosting,0.744188,0.010890,0.660763,0.013023,0.448566,0.973597,1.289585


In [10]:
winner_name = results.index[0]

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
    search = tune_winner(winner_name, lookback_days, random_state, X_train, y_train, groups, cv)

pd.DataFrame(search.cv_results_)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_classifier__C,param_classifier__l1_ratio,param_classifier__solver,params,split0_test_roc_auc,split1_test_roc_auc,...,std_test_pr_auc,rank_test_pr_auc,split0_test_brier,split1_test_brier,split2_test_brier,split3_test_brier,split4_test_brier,mean_test_brier,std_test_brier,rank_test_brier
0,0.112681,0.010251,0.016335,0.002402,0.001000,0.0,saga,"{'classifier__C': 0.001, 'classifier__l1_ratio...",0.755652,0.760226,...,0.006411,5,-0.198788,-0.197465,-0.198592,-0.195078,-0.191768,-0.196338,0.002639,32
1,0.119591,0.002046,0.017777,0.003405,0.001000,0.5,saga,"{'classifier__C': 0.001, 'classifier__l1_ratio...",0.752278,0.763894,...,0.006339,8,-0.202344,-0.200195,-0.202396,-0.199364,-0.196649,-0.200190,0.002132,35
2,0.082423,0.004345,0.018341,0.002704,0.001000,1.0,saga,"{'classifier__C': 0.001, 'classifier__l1_ratio...",0.745484,0.762488,...,0.006995,28,-0.208452,-0.206253,-0.208890,-0.204736,-0.203646,-0.206395,0.002039,36
3,0.078519,0.005100,0.016018,0.003034,0.002848,0.0,saga,"{'classifier__C': 0.002848035868435802, 'class...",0.755812,0.759815,...,0.007294,7,-0.198226,-0.197350,-0.198168,-0.194593,-0.190646,-0.195796,0.002895,31
4,0.109819,0.008105,0.016001,0.003328,0.002848,0.5,saga,"{'classifier__C': 0.002848035868435802, 'class...",0.754315,0.764093,...,0.006142,1,-0.199132,-0.196453,-0.199246,-0.195682,-0.191749,-0.196452,0.002746,33
5,0.109438,0.009611,0.019758,0.002236,0.002848,1.0,saga,"{'classifier__C': 0.002848035868435802, 'class...",0.750013,0.766683,...,0.006545,4,-0.200846,-0.196569,-0.201056,-0.196170,-0.192969,-0.197522,0.003066,34
6,0.084751,0.011851,0.019227,0.001816,0.008111,0.0,saga,"{'classifier__C': 0.008111308307896872, 'class...",0.755733,0.760673,...,0.008127,10,-0.197941,-0.196799,-0.197710,-0.194409,-0.189856,-0.195343,0.003015,27
7,0.096346,0.006608,0.017550,0.003085,0.008111,0.5,saga,"{'classifier__C': 0.008111308307896872, 'class...",0.755603,0.762244,...,0.007041,3,-0.198264,-0.196384,-0.198649,-0.194338,-0.190209,-0.195569,0.003087,29
8,0.093638,0.010866,0.016031,0.002015,0.008111,1.0,saga,"{'classifier__C': 0.008111308307896872, 'class...",0.753317,0.764485,...,0.006398,2,-0.199056,-0.195647,-0.199289,-0.194494,-0.190456,-0.195788,0.003258,30
9,0.130323,0.006800,0.018481,0.005516,0.023101,0.0,saga,"{'classifier__C': 0.02310129700083159, 'classi...",0.755320,0.762470,...,0.009460,11,-0.197897,-0.195917,-0.197179,-0.194403,-0.189400,-0.194959,0.003023,24


## **Model Evaluation**

In [11]:
best_estimator = search.best_estimator_

y_proba = best_estimator.predict_proba(X_test)[:, 1]

y_test = y_test.to_numpy()

threshold = select_threshold(
    y_true=y_test,
    y_proba=y_proba,
    n_scored=int(y_test.size),
    cost=config.economics.cost_per_contact,
    customer_lifetime_value=config.economics.average_customer_lifetime_value,
    save_rate=config.economics.save_rate,
    sensitivity_spread=config.economics.sensitivity_spread,
    sensitivity_points=config.economics.sensitivity_points,
    capacity=config.economics.campaign_capacity
)

baselines = evaluate_baselines(
    X_test=X_test,
    y_test=y_test,
    capacity=config.economics.campaign_capacity,
    random_state=config.model.random_state,
    heuristic_baseline=config.model.heuristic_baseline,
    n_scored=int(y_test.size)
)
cv_metrics = generate_cv_metrics(
    search=search,
    y_train=y_train.to_numpy()
)
test_metrics = evaluate_on_test(
    y_test=y_test,
    y_proba=y_proba,
    threshold=threshold["chosen_threshold"],
)
unseen_test_metrics = evaluate_on_unseen_customers(
    df_processed=df,
    test_cutoff="2011-07-12",
    y_test=y_test,
    y_proba=y_proba
)

build_comparison_table(
    cv_metrics=cv_metrics,
    test_metrics=test_metrics,
    unseen_test_metrics=unseen_test_metrics,
    baselines=baselines
)

,kind,score_type,split,n_rows,n_positive,base_rate,roc_auc,pr_auc,brier,mean_predicted,threshold,n_flagged,precision,recall,f1,lift,note
Random,random,ranking,test,2546,775,0.304399,0.521560,0.316561,None,None,0.806399,500.0,0.296000,0.190968,0.232157,0.972408,"uniform noise, expect roc_auc near 0.50"
Recency (Ranked),rule,ranking,test,2546,775,0.304399,0.644505,0.426016,None,None,98.000000,500.0,0.478000,0.308387,0.374902,1.570307,recency as a continuous ranking
Recency Above 75d (Heuristic),rule,binary,test,2546,775,0.304399,NaN,NaN,None,None,0.500000,759.0,0.459816,0.450323,0.455020,1.510568,"recency_above_75d; hard 0/1, ranking metrics n..."
CV,model,probability,cv,15311,6868,0.448566,0.765248,0.679364,0.19474,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"5-fold, std 0.0073"
Test Dataset,model,probability,test,2546,775,0.304399,0.785558,0.581197,0.172773,0.373088,0.637894,500.0,0.624000,0.402581,0.489412,2.049941,NaN
Test Dataset with Unseen Customers only,model,probability,test_unseen,719,368,0.511822,0.680316,0.659173,0.230804,0.604837,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
feature_names = best_estimator.named_steps["preprocessor"].get_feature_names_out()

best_classifier = best_estimator.named_steps["classifier"]

if hasattr(best_classifier, "feature_importances_"):
    importances = best_classifier.feature_importances_
elif hasattr(best_classifier, "coef_"):
    importances = best_classifier.coef_[0] 
else:
    raise AttributeError("The final estimator has no feature importances or coefficients.")

feature_comparison = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

print(feature_comparison.head(10))

                                          Feature  Importance
25  numerical_log__missingindicator_recency_ratio    0.655975
5              numerical__order_momentum_90d_150d    0.366366
4               numerical__order_momentum_30d_90d    0.326737
20                     numerical_log__orders_150d    0.305463
28            categorical__country_United Kingdom    0.279571
12          numerical_log__avg_inter_purchase_gap    0.197320
29        categorical__country_infrequent_sklearn    0.194940
14                   numerical_log__recency_ratio    0.179379
0                              numerical__recency    0.151886
13          numerical_log__std_inter_purchase_gap    0.108327
